In [0]:
dbutils.widgets.text(name="env", defaultValue="", label="environment")
env = dbutils.widgets.get("env")

In [0]:
%run "./commons"

In [0]:
checkpoint_path


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, current_timestamp, col

schema = StructType([
    StructField("PatientID",StringType()),
    StructField("FirstName", StringType()),
    StructField("LastName",StringType()),
    StructField("MiddleName",StringType()),
    StructField("SSN",StringType()),
    StructField("PhoneNumber",StringType()),
    StructField("Gender",StringType()),
    StructField("DOB",StringType()),
    StructField("Address",StringType()),
    StructField("ModifiedDate",StringType())
])

def write_bronze_data(environment):
    #raw_df = spark.readStream.format("cloudFiles").option("cloudFiles.format","csv").option("cloudFiles.schemaLocation", f'{checkpoint_path}/schemaInfer/').option("cloudFiles.schemaEvolutionMode", "addNewColumns").option("cloudFiles.inferColumnTypes", "true").load(landing_path).withColumn("Extract_Time", current_timestamp().cast("string"))

    raw_df = spark.readStream.format("cloudFiles").option("cloudFiles.format","csv").schema(schema).option("cloudFiles.schemaLocation", f'{checkpoint_path}/schemaInfer/').option("header",True).load(landing_path).withColumn("Extract_Time", current_timestamp().cast("string"))    

    raw_df = raw_df.withColumn("filename", col("_metadata.file_path"))
    
    write_stream = raw_df.writeStream.format("delta").option("checkpointLocation",checkpoint_path).outputMode("append").queryName("raw_patient_load").trigger(availableNow=True).option("mergeSchema", "false").toTable(f"`{environment}_catalog`.`bronze`.`raw_patient`")

    write_stream.awaitTermination()
    print("raw patient records loaded to Bronze layer")

    

write_bronze_data(env)

